# Fused RMSNorm & Residual Add in Triton
Colab notebook for the GPU environment

_*Use a GPU runtime_

Clone the repository

In [ ]:
import os
import shutil

repo_url = "https://github.com/jarnesino/fused-rmsnorm-residual-add-triton.git"
repo_dir = "/content/fused-rmsnorm-residual-add-triton"

# Remove existing clone
%cd /content
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

# Clone again
!git clone "$repo_url" "$repo_dir"

Set up the environment

In [ ]:
%cd /content/fused-rmsnorm-residual-add-triton
!curl -LsSf https://astral.sh/uv/install.sh | sh
!pip install -q go-task-bin
!task gpu:sync
!task gpu:test

In [ ]:
os.environ["MPLBACKEND"] = "agg"

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image as ImageDisplay
from PIL import Image

Run tutorial benchmarks

In [ ]:
!task gpu:bench:tutorials

In [ ]:
ImageDisplay("benchmarks/results/softmax/softmax-bandwidth.png")

Run RMSNormResidualAdd benchmarks

In [ ]:
!TRITON_PRINT_AUTOTUNING=1 task gpu:bench -- --runs 3

In [ ]:
base = Path("benchmarks/results/tesla-t4")
latest = max(p for p in base.iterdir() if p.is_dir())

paths = [
    str(latest / f"forward_{axis}_{dtype}.png")
    for axis in ("N", "M")
    for dtype in ("bfloat16", "float16", "float32")
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, path in zip(axes.flat, paths, strict=True):
    ax.imshow(Image.open(path))
    ax.set_title(path.split("/")[-1].replace(".png", ""))
    ax.axis("off")

plt.tight_layout()
plt.show()

Download results

In [ ]:
!cd benchmarks/results && zip -qr /content/results-tesla-t4.zip tesla-t4
from google.colab import files

files.download("/content/results-tesla-t4.zip")

# PTX and register inspection

Machine data

In [ ]:
%%bash
cd "${REPOSITORY:-/content/fused-rmsnorm-residual-add-triton}"
uv run python - <<'PY'
import subprocess
import torch
import triton

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
dirty = subprocess.run(
    ["git", "status", "--porcelain", "--untracked-files=no"], capture_output=True, text=True
).stdout.strip()

print("gpu:     ", torch.cuda.get_device_name(0))
print("torch:   ", torch.__version__)
print("triton:  ", triton.__version__)
print("cuda:    ", torch.version.cuda)
print("commit:  ", commit, "(dirty)" if dirty else "")
PY

## 1. Vector widths in the PTX

The autotuner compiles every config that survives pruning, and each compiled
config is dumped to its own directory, so the grep below prints one block per
config (not per kernel).

Is the config vectorizing?
- `ld.global.v4.b32` / `st.global.v4.b32` means 16 bytes
- `ld.global.v2.b32` / `st.global.v2.b32` means 8 bytes
- `.b16` means no vectorization

In [ ]:
%%bash
cd "${REPOSITORY:-/content/fused-rmsnorm-residual-add-triton}"

DUMP_DIR="${TRITON_DUMP_DIR:-/content/dump}"
rm -rf "$DUMP_DIR"

export TRITON_KERNEL_DUMP=1
export TRITON_DUMP_DIR="$DUMP_DIR"
export TRITON_ALWAYS_COMPILE=1

uv run python - <<'PY'
import torch
from fused_rmsnorm_residual_add.fused import FusedImplementation

number_of_rows, number_of_columns = 4096, 4096
x = torch.randn(number_of_rows, number_of_columns, device="cuda", dtype=torch.bfloat16)
residual = torch.randn_like(x)
weight = torch.randn(number_of_columns, device="cuda", dtype=torch.bfloat16)

FusedImplementation(weight, 1e-6).forward(x, residual)
PY

for directory in "$DUMP_DIR"/*/; do
  echo "== $(basename "$directory")"
  grep -o 'ld\.global[a-z0-9.]*\|st\.global[a-z0-9.]*' "$directory"*.ptx \
    | sort | uniq -c | sort -rn
done

## 2. Register pressure and the prune ceiling

N=16384 has the whole range the prune has to cover, with 512 elements per thread
at 1 warp, down to 16 at 32 warps.

`n_spills` means local-memory bytes per thread.

When does loweing the number of warps stop getting wider loads?

In [ ]:
%%bash
cd "${REPOSITORY:-/content/fused-rmsnorm-residual-add-triton}"
uv run python - <<'PY'
import torch
from fused_rmsnorm_residual_add import fused

number_of_rows, number_of_columns = 1024, 16384
x = torch.randn(number_of_rows, number_of_columns, device="cuda", dtype=torch.bfloat16)
residual = torch.randn_like(x)
weight = torch.randn(number_of_columns, device="cuda", dtype=torch.bfloat16)

fused.FusedImplementation(weight, 1e-6).forward(x, residual)

kernel = fused.fused_rmsnorm_residual_add_kernel
try:
    device_cache = kernel.fn.device_caches[torch.cuda.current_device()][0]
except (AttributeError, KeyError, IndexError) as error:
    raise RuntimeError(
        "Could not reach Triton's compiled-kernel cache. This relies on Triton "
        "internals, verified against triton 3.6.0. Try: "
        "cuobjdump -res-usage on the compiled binary."
    ) from error

compiled_by_warps = {
    compiled.metadata.num_warps: compiled for compiled in device_cache.values()
}

block_size = number_of_columns
print(f"{'warps':>5} {'elems/thread':>13} {'registers':>10} {'spill bytes':>12}")
for number_of_warps, compiled in sorted(compiled_by_warps.items()):
    elements_per_thread = block_size // (32 * number_of_warps)
    print(
        f"{number_of_warps:>5} {elements_per_thread:>13} "
        f"{compiled.n_regs:>10} {compiled.n_spills:>12}"
    )
PY

`elements_per_thread = BLOCK_N / (32 × num_warps)`

Below 4 warps each thread holds too much state and spills to local memory, when the point is avoiding memory traffic. Above 16 warps each thread no longer holds 16 contiguous bytes, so the compiler drops from ld.global.v4.b32 to 8-byte accesses. The 4-warp row only avoids spilling by taking all 255 registers the hardware allows, which leaves almost no room for concurrent warps. The autotuner's picks is in the middle.